# 📚 SQL Ch.4 — Subquery & CTE
> BigQuery SQL Reference Guide, Chapter 4: Inline/Scalar/FROM Subqueries · EXISTS · WITH (CTE) · UNION  
> BigQuery SQL 완전 참조 가이드 4장: 인라인/스칼라/FROM 서브쿼리 · EXISTS · WITH (CTE) · UNION

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Write a query-inside-a-query to compare rows against a computed value (like the average) instead of a hardcoded number  
하드코딩된 숫자 대신 평균 같은 계산값과 행을 비교하는 쿼리 안의 쿼리를 작성한다
- [x] Explain why `NOT IN` can silently return zero rows when the subquery contains a `NULL`, and use `NOT EXISTS` instead  
서브쿼리에 `NULL`이 있으면 `NOT IN`이 왜 조용히 0행을 반환할 수 있는지 설명하고 `NOT EXISTS`를 대신 사용한다
- [x] Use `WITH` (CTE) to break a multi-step calculation into named, readable stages  
`WITH`(CTE)로 여러 단계짜리 계산을 이름 붙인 읽기 쉬운 단계로 나눈다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** A subquery is a `SELECT` nested inside another query — its result gets used by the outer query instead of a literal value or table. A CTE (`WITH ... AS (...)`) does the same job as a subquery but gives it a name up front, so a multi-step calculation reads top-to-bottom like a recipe instead of being buried in parentheses. Both let you break "get this number, then use it" into two clean, verifiable stages.

**KR:** 서브쿼리는 다른 쿼리 안에 중첩된 `SELECT`이며, 그 결과가 리터럴 값이나 테이블 대신 바깥 쿼리에서 사용됩니다. CTE(`WITH ... AS (...)`)는 서브쿼리와 같은 일을 하지만 미리 이름을 붙여두므로, 여러 단계짜리 계산이 괄호 속에 파묻히는 대신 위에서 아래로 레시피처럼 읽힙니다. 둘 다 "이 값을 구하고, 그다음 그 값을 쓴다"는 작업을 두 개의 깔끔하고 검증 가능한 단계로 나누게 해줍니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Hardcoding a number like "average = 52,200" into a query works once — then the data changes and the number silently becomes wrong, with no error to warn you. A subquery computes that value fresh every time the query runs, so it's always correct for the current data, and multi-step logic (filter → aggregate → filter the aggregate) gets a place to live instead of being crammed into one impossible-to-read line.

**KR:** "평균 = 52,200"처럼 숫자를 쿼리에 하드코딩하면 한 번은 동작하지만, 이후 데이터가 바뀌면 경고 하나 없이 조용히 틀린 값이 됩니다. 서브쿼리는 쿼리가 실행될 때마다 그 값을 새로 계산하므로 항상 현재 데이터에 맞고, 여러 단계짜리 로직(필터 → 집계 → 집계 결과를 다시 필터)이 읽기 불가능한 한 줄에 욱여넣어지는 대신 제자리를 갖게 됩니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** "Customers who spent more than average," "products that have never sold," "this month vs. every prior month's average" — all of these compare something to a value that has to be *computed from the data itself*, which is exactly what a subquery or CTE is for. CTEs in particular are how BA analysts write long, auditable queries: name each stage, and anyone reading it later (including future-you) can follow the logic stage by stage.

**KR:** "평균보다 많이 쓴 고객", "한 번도 안 팔린 상품", "이번 달 vs 이전 모든 달 평균" — 모두 *데이터 자체에서 계산해야 하는* 값과 무언가를 비교하는 것이며, 이게 바로 서브쿼리나 CTE가 필요한 이유입니다. 특히 CTE는 BA가 길고 검증 가능한 쿼리를 작성하는 방법입니다: 각 단계에 이름을 붙이면, 나중에 읽는 누구든(미래의 나 자신을 포함해서) 단계별로 로직을 따라갈 수 있습니다.

**Comparison / 비교표:**

| Task / 작업 | SQL | Pandas |
|---|---|---|
| Compare to a computed value / 계산값과 비교 | Scalar subquery in `WHERE` | compute first, then `df[df.col > value]` |
| Check membership in a list / 목록 포함 여부 | `IN (subquery)` | `df[df.col.isin(other_df.col)]` |
| Check membership safely (NULL-proof) / 안전한 포함 확인 | `EXISTS` / `NOT EXISTS` | `.isin()` (NULL-safe, unlike SQL's `NOT IN`) |
| Name an intermediate result / 중간 결과에 이름 붙이기 | `WITH name AS (...)` | assign to a variable |
| Stack two tables / 두 테이블 쌓기 | `UNION ALL` / `UNION DISTINCT` | `pd.concat()` (+ `.drop_duplicates()`) |

---
# 📝 Syntax

## Basic Syntax
An inline subquery inside `WHERE` — the parentheses run first, produce one value, and that value feeds the comparison.  
`WHERE` 안의 인라인 서브쿼리 — 괄호 안이 먼저 실행되어 값 하나를 만들고, 그 값이 비교에 쓰입니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

orders = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005],
    "customer_id": ["C01", "C02", "C01", "C03", "C02"],
    "amount":      [45000, 32000, 61000, 28000, 95000],
})

print("-- ❌ hardcoded: works once, then silently goes stale as data changes --")
print("-- ❌ 하드코딩: 한 번은 동작하지만 데이터가 바뀌면 조용히 틀려짐 --")
display(run("SELECT order_id, amount FROM orders WHERE amount > 52200"))

print("-- ✅ subquery: the average is recomputed every run, always correct --")
print("-- ✅ 서브쿼리: 평균이 매번 다시 계산되어 항상 정확함 --")
display(run("SELECT order_id, amount FROM orders WHERE amount > (SELECT AVG(amount) FROM orders)"))
# Both return the same 2 rows here -- but only the subquery version stays correct after the data changes.
# 지금은 둘 다 같은 2행을 반환하지만, 데이터가 바뀐 뒤에도 정확한 건 서브쿼리 버전뿐.


-- ❌ hardcoded: works once, then silently goes stale as data changes --
-- ❌ 하드코딩: 한 번은 동작하지만 데이터가 바뀌면 조용히 틀려짐 --


,order_id,amount
0,1003,61000
1,1005,95000


-- ✅ subquery: the average is recomputed every run, always correct --
-- ✅ 서브쿼리: 평균이 매번 다시 계산되어 항상 정확함 --


,order_id,amount
0,1003,61000
1,1005,95000


## Common Variations

In [2]:
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "name":        ["김민수", "이영희", "박준호", "최서연"],
})

# IN (subquery) -- compares against a LIST of values instead of one value
# IN (서브쿼리) -- 값 하나가 아니라 값의 "목록"과 비교
sql = """
SELECT name
FROM customers
WHERE customer_id IN (
    SELECT customer_id FROM orders WHERE amount > 50000
)
"""
display(run(sql))
# pandas equivalent: df[df["customer_id"].isin(other_df["customer_id"])]


,name
0,김민수
1,이영희


---
# 🧪 Small Examples

## Example 1 — Scalar Subquery: A Single Value Inside SELECT / SELECT절 안에서 단일 값 반환
**EN:** A subquery can also sit inside `SELECT` itself, computing one value that gets attached to every row — like broadcasting the overall average onto each row so you can compare each row against it directly. **Rule:** a scalar subquery must return exactly one row and one column, or the query errors.  
**KR:** 서브쿼리는 `SELECT` 안에도 들어갈 수 있으며, 모든 행에 붙는 값 하나를 계산합니다 — 전체 평균을 각 행에 브로드캐스트해서 바로 비교할 수 있게 하는 식입니다. **규칙:** 스칼라 서브쿼리는 정확히 1행 1열만 반환해야 하며, 그렇지 않으면 오류가 납니다.

In [3]:
sql = """
SELECT
    order_id,
    amount,
    (SELECT AVG(amount) FROM orders) AS avg_amount,
    amount - (SELECT AVG(amount) FROM orders) AS diff_from_avg
FROM orders
"""
display(run(sql))
# pandas equivalent: df["diff_from_avg"] = df["amount"] - df["amount"].mean()


,order_id,amount,avg_amount,diff_from_avg
0,1001,45000,52200.0,-7200.0
1,1002,32000,52200.0,-20200.0
2,1003,61000,52200.0,8800.0
3,1004,28000,52200.0,-24200.0
4,1005,95000,52200.0,42800.0


## Example 2 — FROM-Clause Subquery: Using a Query's Result as a Table / 서브쿼리 결과를 테이블처럼 사용
**EN:** A subquery can also replace a table name in `FROM` — the inner query runs first and builds a temporary result, and the outer query then filters/selects from *that* result as if it were a real table. This is how you filter on an aggregate without duplicating the `GROUP BY` logic in a `HAVING` clause. A `FROM`-subquery **must** have an alias (like `AS region_totals`) — it has no name of its own.  
**KR:** 서브쿼리는 `FROM`에서 테이블 이름 자리를 대신할 수도 있습니다 — 안쪽 쿼리가 먼저 실행되어 임시 결과를 만들고, 바깥 쿼리는 *그 결과*를 마치 진짜 테이블처럼 필터링/선택합니다. `HAVING` 절에 `GROUP BY` 로직을 중복시키지 않고 집계값을 필터링하는 방법입니다. `FROM` 서브쿼리는 **반드시** 별칭(예: `AS region_totals`)이 있어야 합니다 — 자기 자신의 이름이 없기 때문입니다.

In [4]:
orders3 = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "region":   ["서울", "부산", "서울", "인천", "부산"],
    "amount":   [45000, 32000, 61000, 28000, 95000],
})

sql = """
SELECT region, total
FROM (
    SELECT region, SUM(amount) AS total
    FROM orders3
    GROUP BY region
) AS region_totals
WHERE total >= 100000
ORDER BY total DESC
"""
display(run(sql))
# The inner query builds a (region, total) summary table first;
# the outer query then filters that summary -- same result as GROUP BY + HAVING,
# but the subquery form is easier to extend with further joins/calculations.
# 안쪽 쿼리가 먼저 (region, total) 요약 테이블을 만들고,
# 바깥 쿼리가 그 요약을 다시 필터링 -- GROUP BY + HAVING과 결과는 같지만,
# 서브쿼리 형태는 이후 JOIN이나 추가 계산으로 확장하기가 더 쉬움.


,region,total
0,부산,127000.0
1,서울,106000.0


## Example 3 — EXISTS / NOT EXISTS: The NULL-Safe Way to Check Existence / 존재 여부 확인
**EN:** `EXISTS (subquery)` is `TRUE` if the subquery returns *at least one row*, `FALSE` if it returns none — the actual values don't matter, only whether a row exists, so `SELECT 1` is the conventional placeholder. `NOT EXISTS` is its opposite.  
**KR:** `EXISTS (서브쿼리)`는 서브쿼리가 *행을 하나라도* 반환하면 `TRUE`, 하나도 없으면 `FALSE`입니다 — 실제 값은 중요하지 않고 행이 존재하는지만 중요하므로 `SELECT 1`이 관례적인 자리표시자입니다. `NOT EXISTS`는 그 반대입니다.

⚠️ **EN:** `NOT IN` and `NOT EXISTS` look interchangeable, but they are **not** when the subquery can contain a `NULL`. This is one of the most dangerous silent bugs in SQL — watch closely below.  
⚠️ **KR:** `NOT IN`과 `NOT EXISTS`는 서로 바꿔 써도 될 것 같지만, 서브쿼리에 `NULL`이 있을 수 있다면 **절대 그렇지 않습니다**. SQL에서 가장 위험한 조용한 버그 중 하나입니다 — 아래를 잘 보세요.

In [5]:
customers4 = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "name":        ["김민수", "이영희", "박준호", "최서연"],
})
orders4 = pd.DataFrame({"order_id": [1001, 1002], "customer_id": ["C01", "C02"], "amount": [45000, 32000]})

print("-- EXISTS: customers who HAVE ordered / 주문 이력이 있는 고객 --")
display(run("SELECT name FROM customers4 c WHERE EXISTS (SELECT 1 FROM orders4 o WHERE o.customer_id=c.customer_id)"))

print("-- NOT EXISTS: customers who have NEVER ordered / 주문 이력이 없는 고객 --")
display(run("SELECT name FROM customers4 c WHERE NOT EXISTS (SELECT 1 FROM orders4 o WHERE o.customer_id=c.customer_id)"))

print()
print("=" * 60)
print("⚠️ THE TRAP: same question, but the subquery now contains a NULL customer_id")
print("⚠️ 함정: 같은 질문이지만, 서브쿼리에 NULL customer_id가 섞여 있음")
orders_null = pd.DataFrame({
    "order_id":    [1001, 1002, 1003],
    "customer_id": ["C01", None, "C02"],   # <- a NULL customer_id, e.g. a bad data entry
    "amount":      [45000, 32000, 61000],
})

print("-- ❌ NOT IN: silently returns 0 rows, even though 박준호/최서연 clearly never ordered --")
print("-- ❌ NOT IN: 박준호·최서연이 분명 주문한 적 없는데도 조용히 0행 반환 --")
r1 = run("SELECT name FROM customers4 c WHERE c.customer_id NOT IN (SELECT customer_id FROM orders_null)")
display(r1)
print(f"rows: {len(r1)}  (expected 2, got {len(r1)} -- NOT IN silently broke)")

print("-- ✅ NOT EXISTS: correctly ignores the NULL, gives the right answer --")
print("-- ✅ NOT EXISTS: NULL과 무관하게 정확히 동작 --")
r2 = run("SELECT name FROM customers4 c WHERE NOT EXISTS (SELECT 1 FROM orders_null o WHERE o.customer_id = c.customer_id)")
display(r2)
# Why: "x NOT IN (A, NULL, B)" expands to "x!=A AND x!=NULL AND x!=B". Any comparison to NULL
# evaluates to UNKNOWN, and UNKNOWN inside an AND chain poisons the whole condition to UNKNOWN --
# so EVERY row gets excluded, regardless of whether it actually matched anything.
# 이유: "x NOT IN (A, NULL, B)"는 "x!=A AND x!=NULL AND x!=B"로 풀림. NULL과의 비교는 항상
# UNKNOWN이 되고, AND 체인 안에 UNKNOWN이 하나라도 있으면 조건 전체가 UNKNOWN이 됨 --
# 그래서 실제 매칭 여부와 무관하게 모든 행이 제외됨.


-- EXISTS: customers who HAVE ordered / 주문 이력이 있는 고객 --


,name
0,김민수
1,이영희


-- NOT EXISTS: customers who have NEVER ordered / 주문 이력이 없는 고객 --


,name
0,박준호
1,최서연



⚠️ THE TRAP: same question, but the subquery now contains a NULL customer_id
⚠️ 함정: 같은 질문이지만, 서브쿼리에 NULL customer_id가 섞여 있음
-- ❌ NOT IN: silently returns 0 rows, even though 박준호/최서연 clearly never ordered --
-- ❌ NOT IN: 박준호·최서연이 분명 주문한 적 없는데도 조용히 0행 반환 --


,name


rows: 0  (expected 2, got 0 -- NOT IN silently broke)
-- ✅ NOT EXISTS: correctly ignores the NULL, gives the right answer --
-- ✅ NOT EXISTS: NULL과 무관하게 정확히 동작 --


,name
0,박준호
1,최서연


## Example 4 — WITH (CTE): Naming a Subquery / WITH (CTE) — 서브쿼리에 이름 붙이기
**EN:** A CTE (Common Table Expression) is the exact same idea as Example 2's `FROM`-subquery, just written differently: define it once at the top with `WITH name AS (...)`, then reference `name` in the main query below like a real table. Same result, but the "compute this, then use it" structure is visible in the code's shape, not buried in nested parentheses.  
**KR:** CTE(공통 테이블 표현식)는 예제 2의 `FROM` 서브쿼리와 완전히 같은 개념이며, 쓰는 방식만 다릅니다: 맨 위에서 `WITH name AS (...)`로 한 번 정의하고, 아래 메인 쿼리에서 `name`을 진짜 테이블처럼 참조합니다. 결과는 같지만, "이걸 계산하고, 그다음 이걸 쓴다"는 구조가 중첩된 괄호 속에 숨는 대신 코드 모양에서 바로 드러납니다.

In [6]:
print("-- FROM-subquery version (Example 2, for comparison) --")
sql_sub = """
SELECT region, total
FROM (SELECT region, SUM(amount) AS total FROM orders3 GROUP BY region) AS region_totals
WHERE total >= 100000
ORDER BY total DESC
"""
display(run(sql_sub))

print("-- CTE version -- same result, top-to-bottom reading order --")
print("-- CTE 버전 -- 결과는 동일, 위에서 아래로 읽는 순서 --")
sql_cte = """
WITH region_totals AS (
    SELECT region, SUM(amount) AS total
    FROM orders3
    GROUP BY region
)
SELECT region, total
FROM region_totals
WHERE total >= 100000
ORDER BY total DESC
"""
display(run(sql_cte))
# pandas equivalent: region_totals = df.groupby("region")["amount"].sum().reset_index()
#                     region_totals[region_totals["amount"] >= 100000]


-- FROM-subquery version (Example 2, for comparison) --


,region,total
0,부산,127000.0
1,서울,106000.0


-- CTE version -- same result, top-to-bottom reading order --
-- CTE 버전 -- 결과는 동일, 위에서 아래로 읽는 순서 --


,region,total
0,부산,127000.0
1,서울,106000.0


## Example 5 — Multiple CTEs: Chaining Named Stages / 다중 CTE — 여러 WITH 절 연결
**EN:** Define several CTEs by separating them with commas — each becomes its own named stage, and later CTEs can reference earlier ones (though they don't have to). A one-row CTE (like an overall average) combined with `CROSS JOIN` is a common pattern for attaching one shared reference value to every row of another CTE.  
**KR:** 쉼표로 구분해서 여러 CTE를 정의할 수 있으며, 각각이 이름 붙은 자기만의 단계가 됩니다 — 뒤에 오는 CTE는 앞의 CTE를 참조할 수 있습니다(꼭 그럴 필요는 없지만). 1행짜리 CTE(예: 전체 평균)를 `CROSS JOIN`으로 붙이는 것은 다른 CTE의 모든 행에 하나의 공유 기준값을 부여하는 흔한 패턴입니다.

In [7]:
orders6 = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "region":   ["서울", "부산", "서울", "인천", "부산", "서울"],
    "amount":   [45000, 32000, 61000, 28000, 95000, 30000],
})

sql = """
WITH region_avg AS (
    SELECT region, AVG(amount) AS avg_amount
    FROM orders6
    GROUP BY region
),
overall_avg AS (
    SELECT AVG(amount) AS overall_amount
    FROM orders6
)
SELECT
    r.region,
    ROUND(r.avg_amount, 0)    AS region_avg,
    ROUND(o.overall_amount, 0) AS overall_avg
FROM region_avg r
CROSS JOIN overall_avg o        -- overall_avg has exactly 1 row, so this attaches it to every region
WHERE r.avg_amount > o.overall_amount
"""
display(run(sql))
# Only 부산's regional average beats the overall average.
# 부산의 지역 평균만 전체 평균을 웃돎.


,region,region_avg,overall_avg
0,부산,63500.0,48500.0


## Example 6 — UNION vs UNION ALL: Stacking Two Result Sets / 차이와 사용 시점
**EN:** `UNION ALL` stacks two queries' results on top of each other, keeping every row including duplicates — and it's fast, since it doesn't have to check for duplicates. `UNION DISTINCT` does the same stacking but then removes exact duplicate rows — which means comparing every row against every other row, so it's slower on large data. **BigQuery requires you to write `ALL` or `DISTINCT` explicitly** — bare `UNION` is a syntax error there (note: some other engines default bare `UNION` to `DISTINCT`, so always be explicit for portability).  
**KR:** `UNION ALL`은 두 쿼리의 결과를 그대로 쌓아 올리며, 중복을 포함한 모든 행을 유지합니다 — 중복을 검사할 필요가 없어 빠릅니다. `UNION DISTINCT`도 똑같이 쌓지만 완전히 같은 행은 제거합니다 — 모든 행을 다른 모든 행과 비교해야 하므로 데이터가 클수록 느립니다. **BigQuery는 `ALL` 또는 `DISTINCT`를 반드시 명시해야 하며**, `UNION`만 쓰면 문법 오류입니다(참고: 일부 다른 엔진은 `UNION`만 써도 `DISTINCT`로 기본 동작하므로, 이식성을 위해 항상 명시하는 게 좋습니다).

In [8]:
online_orders = pd.DataFrame({"customer_id": ["C01", "C02"], "order_date": ["2024-01-15", "2024-02-01"]})
offline_orders = pd.DataFrame({"customer_id": ["C02", "C03"], "order_date": ["2024-02-01", "2024-03-10"]})
# Note: C02/2024-02-01 appears in BOTH tables -- a duplicate once stacked.
# 참고: C02/2024-02-01은 두 테이블 모두에 있음 -- 합치면 중복.

print("-- UNION ALL: keeps the duplicate, 4 rows / 중복 포함, 4행 --")
display(run("SELECT customer_id, order_date FROM online_orders UNION ALL SELECT customer_id, order_date FROM offline_orders"))

print("-- UNION DISTINCT: merges the duplicate away, 3 rows / 중복 제거, 3행 --")
display(run("SELECT customer_id, order_date FROM online_orders UNION DISTINCT SELECT customer_id, order_date FROM offline_orders"))
# Rule of thumb: default to UNION ALL unless you specifically know duplicates are possible AND unwanted.
# 실무 규칙: 중복이 실제로 생길 수 있고 원치 않는 게 확실할 때가 아니라면 기본은 UNION ALL.


-- UNION ALL: keeps the duplicate, 4 rows / 중복 포함, 4행 --


,customer_id,order_date
0,C01,2024-01-15
1,C02,2024-02-01
2,C02,2024-02-01
3,C03,2024-03-10


-- UNION DISTINCT: merges the duplicate away, 3 rows / 중복 제거, 3행 --


,customer_id,order_date
0,C03,2024-03-10
1,C02,2024-02-01
2,C01,2024-01-15


## Example 7 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** stacks all three tools from this chapter into one query: a CTE computes per-customer totals, a `JOIN` attaches customer names, and a scalar subquery supplies the "average" threshold to compare against. **Pattern B** shows the most common reason to reach for `UNION ALL`: combining two same-shaped tables from different sources (online + offline) before aggregating them together as one.  
**KR:** **패턴 A**는 이번 챕터의 세 가지 도구를 한 쿼리에 쌓습니다: CTE가 고객별 합계를 계산하고, `JOIN`이 고객 이름을 붙이고, 스칼라 서브쿼리가 비교 기준이 될 "평균"을 제공합니다. **패턴 B**는 `UNION ALL`을 쓰는 가장 흔한 이유를 보여줍니다: 서로 다른 출처(온라인 + 오프라인)의 모양이 같은 두 테이블을 하나로 합쳐서 함께 집계하는 것.

In [9]:
customers7 = pd.DataFrame({"customer_id": ["C01", "C02", "C03"], "name": ["김민수", "이영희", "박준호"]})
orders7 = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "customer_id": ["C01", "C02", "C01", "C03", "C02"],
    "amount": [45000, 32000, 61000, 28000, 95000],
})

print("-- Pattern A: CTE + JOIN + scalar subquery -- above-average-spend customers --")
print("-- 패턴 A: CTE + JOIN + 스칼라 서브쿼리 -- 평균 이상 구매 고객 --")
sql_a = """
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total FROM orders7 GROUP BY customer_id
)
SELECT c.name, ct.total
FROM customer_totals ct
JOIN customers7 c ON ct.customer_id = c.customer_id
WHERE ct.total > (SELECT AVG(total) FROM customer_totals)
ORDER BY ct.total DESC
"""
display(run(sql_a))

print("-- Pattern B: UNION ALL + CTE + GROUP BY -- merge two channels, then total --")
print("-- 패턴 B: UNION ALL + CTE + GROUP BY -- 여러 채널 데이터 합쳐서 집계 --")
online2 = pd.DataFrame({"customer_id": ["C01", "C02"], "amount": [30000, 50000]})
offline2 = pd.DataFrame({"customer_id": ["C02", "C03"], "amount": [20000, 40000]})
sql_b = """
WITH all_orders AS (
    SELECT customer_id, amount FROM online2
    UNION ALL
    SELECT customer_id, amount FROM offline2
)
SELECT customer_id, SUM(amount) AS total
FROM all_orders
GROUP BY customer_id
ORDER BY total DESC
"""
display(run(sql_b))


-- Pattern A: CTE + JOIN + scalar subquery -- above-average-spend customers --
-- 패턴 A: CTE + JOIN + 스칼라 서브쿼리 -- 평균 이상 구매 고객 --


,name,total
0,이영희,127000.0
1,김민수,106000.0


-- Pattern B: UNION ALL + CTE + GROUP BY -- merge two channels, then total --
-- 패턴 B: UNION ALL + CTE + GROUP BY -- 여러 채널 데이터 합쳐서 집계 --


,customer_id,total
0,C02,70000.0
1,C03,40000.0
2,C01,30000.0


## Example 8 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `SELECT` `EXISTS` `WITH` `DESC`
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `SELECT` `EXISTS` `WITH` `DESC`

In [12]:
products_p = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04"],
    "name":       ["노트북", "마우스", "키보드", "모니터"],
    "price":      [1200000, 25000, 45000, 350000],
})
order_items_p = pd.DataFrame({
    "order_id":   [5001, 5002, 5003, 5004],
    "product_id": ["P01", "P02", "P01", "P04"],
    "quantity":   [2, 5, 1, 4],
})
# Note: P03 (키보드) never appears in order_items_p / P03(키보드)는 order_items_p에 한 번도 등장하지 않음

# Q1. Names of products priced above the average price (use a subquery).
# Q1. 평균 가격보다 비싼 제품의 이름 (서브쿼리 사용).
q1 = """
SELECT name
FROM products_p
WHERE price > (SELECT AVG(price) FROM products_p)
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. Names of products that have NEVER been ordered (use the EXISTS family).
# Q2. 한 번도 주문되지 않은 제품의 이름 (EXISTS 계열 사용).
q2 = """
SELECT name
FROM products_p p
WHERE NOT EXISTS (
    SELECT 1 FROM order_items_p oi WHERE oi.product_id = p.product_id
)
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Use a CTE to get total quantity sold per product, joined to product names, highest first.
# Q3. CTE로 제품별 총 판매 수량을 구하고, 제품 이름과 함께 판매 수량 내림차순으로.
q3 = """
WITH product_qty AS (
    SELECT product_id, SUM(quantity) AS total_qty
    FROM order_items_p
    GROUP BY product_id
)
SELECT p.name, pq.total_qty
FROM product_qty pq
JOIN products_p p ON pq.product_id = p.product_id
ORDER BY pq.total_qty DESC
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,name
0,노트북


,name
0,키보드


,name,total_qty
0,마우스,5.0
1,모니터,4.0
2,노트북,3.0


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT name
FROM products_p
WHERE price > (SELECT AVG(price) FROM products_p)

-- Q2
SELECT name
FROM products_p p
WHERE NOT EXISTS (
    SELECT 1 FROM order_items_p oi WHERE oi.product_id = p.product_id
)

-- Q3
WITH product_qty AS (
    SELECT product_id, SUM(quantity) AS total_qty
    FROM order_items_p
    GROUP BY product_id
)
SELECT p.name, pq.total_qty
FROM product_qty pq
JOIN products_p p ON pq.product_id = p.product_id
ORDER BY pq.total_qty DESC
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Using `NOT IN` when the subquery might contain `NULL`**
- EN: If even one value in a `NOT IN` subquery's result is `NULL`, the *entire* `NOT IN` condition silently evaluates to `UNKNOWN` for every row — the query runs, returns no error, and just quietly gives zero (or too few) rows.
- KR: `NOT IN` 서브쿼리 결과에 `NULL`이 단 하나라도 있으면, `NOT IN` 조건 *전체*가 모든 행에 대해 조용히 `UNKNOWN`으로 평가됩니다 — 쿼리는 실행되고 오류도 없이 그냥 조용히 0행(또는 너무 적은 행)을 반환합니다.
- ✅ Fix / 해결법: If a subquery's result column could ever contain `NULL`, use `NOT EXISTS` instead of `NOT IN` — it's immune to this problem.  
서브쿼리 결과 열에 `NULL`이 있을 가능성이 있다면 `NOT IN` 대신 `NOT EXISTS`를 쓰세요 — 이 문제에서 안전합니다.

**Mistake 2 — Forgetting the alias on a FROM-clause subquery**
- EN: `SELECT * FROM (SELECT region, SUM(amount) FROM orders GROUP BY region) WHERE ...` errors, because a subquery used as a table has no name for the outer query to refer to.
- KR: `SELECT * FROM (SELECT region, SUM(amount) FROM orders GROUP BY region) WHERE ...`는 오류가 납니다. 테이블로 쓰인 서브쿼리에 바깥 쿼리가 참조할 이름이 없기 때문입니다.
- ✅ Fix / 해결법: Always alias a `FROM`-subquery, e.g. `... ) AS region_totals`.  
`FROM` 서브쿼리에는 항상 별칭을 붙이세요, 예: `... ) AS region_totals`.

**Mistake 3 — Expecting a scalar subquery to return more than one row**
- EN: `SELECT amount, (SELECT amount FROM orders) FROM orders` errors at runtime once the inner `orders` has more than one row — a scalar subquery is a contract that promises exactly one value.
- KR: `SELECT amount, (SELECT amount FROM orders) FROM orders`는 안쪽 `orders`가 두 행 이상이 되는 순간 실행 오류가 납니다 — 스칼라 서브쿼리는 정확히 값 하나만 낸다는 약속입니다.
- ✅ Fix / 해결법: Wrap the inner query in an aggregate (`AVG`, `MAX`, `COUNT`...) that's guaranteed to collapse to one row, like `(SELECT AVG(amount) FROM orders)`.  
반드시 한 행으로 줄어드는 집계 함수(`AVG`, `MAX`, `COUNT` 등)로 안쪽 쿼리를 감싸세요, 예: `(SELECT AVG(amount) FROM orders)`.

**Mistake 4 — Referencing a later CTE from an earlier one**
- EN: In `WITH a AS (...), b AS (...)`, `a` cannot reference `b` — CTEs can only look "backwards" at ones defined above them, never "forwards."
- KR: `WITH a AS (...), b AS (...)`에서 `a`는 `b`를 참조할 수 없습니다 — CTE는 자기보다 위에 정의된 것만 "뒤돌아볼" 수 있고, "앞을 내다볼" 수는 없습니다.
- ✅ Fix / 해결법: Order your CTEs so each one only depends on CTEs defined above it — build "bottom up," the same direction you read the query.  
CTE는 자기보다 위에 정의된 CTE에만 의존하도록 순서를 정하세요 — 쿼리를 읽는 방향과 같은 "아래로 쌓는" 순서로 만드세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Make `NOT EXISTS` your default over `NOT IN` — it's never wrong, while `NOT IN` is only safe when you can *guarantee* the subquery column has no `NULL`s.
- `NOT IN` 대신 `NOT EXISTS`를 기본으로 삼으세요 — `NOT EXISTS`는 절대 틀리지 않지만, `NOT IN`은 서브쿼리 열에 `NULL`이 없다고 *확신할 수 있을 때만* 안전합니다.
- Reach for a CTE the moment a query needs more than one mental "step" — readability compounds, especially for a query you'll revisit in three months.
- 쿼리에 머릿속 "단계"가 하나 이상 필요해지는 순간 CTE를 꺼내드세요 — 특히 3개월 뒤 다시 볼 쿼리라면 가독성은 복리로 쌓입니다.
- Default to `UNION ALL`, not `UNION DISTINCT` — it's faster, and duplicate rows across two genuinely different sources are rarer than you'd expect.
- `UNION DISTINCT`가 아니라 `UNION ALL`을 기본으로 쓰세요 — 더 빠르고, 서로 정말 다른 두 출처 사이에 중복 행이 생기는 경우는 생각보다 드뭅니다.
- Test a subquery by itself first (run just the inner `SELECT` alone) before wrapping it in the outer query — it's much easier to debug one small piece at a time.
- 서브쿼리를 바깥 쿼리로 감싸기 전에 그 자체로 먼저 테스트하세요(안쪽 `SELECT`만 따로 실행) — 작은 조각 하나씩 디버깅하는 게 훨씬 쉽습니다.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY
 3. JOIN
 4. Subquery & CTE               ← ★ YOU ARE HERE / 지금 여기
 5. Conditions & NULL Handling
 6. String & Date Functions
 7. Window Functions
 8. BA-Specific Patterns
```

```
Three ways to say "compute this, then use it" / "이걸 계산하고 쓴다"의 세 가지 표현
──────────────────────────────────────────────
  Inline/Scalar subquery   FROM-subquery              CTE
  WHERE x > (SELECT...)    FROM (SELECT...) AS t       WITH t AS (SELECT...)
  -> one value             -> one temp "table"         -> one NAMED temp "table"
  -> 값 하나                -> 임시 "테이블" 하나          -> 이름 붙은 임시 "테이블"
                                                          (same idea, clearer to read)
                                                          (같은 개념, 더 읽기 쉬움)
```

*How is today's topic connected to other concepts?*

**EN:** Chapter 4 leans directly on Chapter 3's `JOIN` (Example 7, Pattern A, joins a CTE back to a real table) and Chapter 2's `GROUP BY` (every CTE in this chapter aggregates first). Nothing about `WHERE`/`GROUP BY`/`JOIN` changes inside a subquery or CTE — they're just query results being treated as tables, so all prior chapters' rules still apply *inside* them. Looking ahead, Chapter 5 (`CASE WHEN`, `NULL` handling) will show up constantly *inside* CTEs, since a CTE is often exactly where you'd first clean or categorize raw data before the main query touches it.

**KR:** 4장은 3장의 `JOIN`(예제 7 패턴 A에서 CTE를 실제 테이블에 다시 JOIN함)과 2장의 `GROUP BY`(이번 챕터의 모든 CTE가 먼저 집계함)에 직접 기대고 있습니다. 서브쿼리나 CTE 안이라고 해서 `WHERE`/`GROUP BY`/`JOIN`이 달라지지 않습니다 — 그냥 쿼리 결과를 테이블처럼 다루는 것뿐이라, 앞선 챕터의 규칙이 *그 안에서도* 그대로 적용됩니다. 앞으로 배울 5장(`CASE WHEN`, `NULL` 처리)은 CTE *안에서* 끊임없이 등장하게 되는데, CTE는 종종 메인 쿼리가 손대기 전에 원본 데이터를 먼저 정제하거나 분류하는 자리이기 때문입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** A product manager asks: *"Which products are selling above the average line-total, and which have never sold at all? I need both lists for the quarterly review."* This needs a scalar subquery for the first list and `NOT EXISTS` for the second — a perfect two-part use of this chapter.
**KR:** 프로덕트 매니저가 묻습니다: *"평균 판매 금액보다 잘 팔리는 제품이 뭐고, 아예 한 번도 안 팔린 제품은 뭐야? 분기 리뷰에 둘 다 필요해."* 첫 번째 목록엔 스칼라 서브쿼리, 두 번째엔 `NOT EXISTS`가 필요합니다 — 이번 챕터를 딱 두 부분으로 나눠 쓰는 완벽한 예입니다.

**To-do / 할 일:**
- [x] Compute each product's total sales via a CTE  
CTE로 제품별 총 판매액을 계산한다
- [x] Compare each product's total against the overall average with a scalar subquery  
스칼라 서브쿼리로 전체 평균과 각 제품의 합계를 비교한다
- [x] Separately, list products with `NOT EXISTS` a matching sale  
별도로 `NOT EXISTS`로 판매 기록이 없는 제품을 나열한다

In [11]:
products_biz = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04", "P05"],
    "name":       ["노트북", "마우스", "키보드", "모니터", "웹캠"],
})
sales_biz = pd.DataFrame({
    "sale_id":    [9001, 9002, 9003, 9004],
    "product_id": ["P01", "P01", "P02", "P04"],
    "line_total": [1200000, 1200000, 25000, 350000],
})

print("-- above-average-selling products / 평균 이상 판매 제품 --")
sql_above = """
WITH product_totals AS (
    SELECT product_id, SUM(line_total) AS total_sales
    FROM sales_biz
    GROUP BY product_id
)
SELECT p.name, pt.total_sales
FROM product_totals pt
JOIN products_biz p ON pt.product_id = p.product_id
WHERE pt.total_sales > (SELECT AVG(total_sales) FROM product_totals)
ORDER BY pt.total_sales DESC
"""
display(run(sql_above))

print("-- never-sold products / 한 번도 안 팔린 제품 --")
sql_never = """
SELECT name
FROM products_biz p
WHERE NOT EXISTS (SELECT 1 FROM sales_biz s WHERE s.product_id = p.product_id)
"""
display(run(sql_never))


-- above-average-selling products / 평균 이상 판매 제품 --


,name,total_sales
0,노트북,2400000.0


-- never-sold products / 한 번도 안 팔린 제품 --


,name
0,키보드
1,웹캠


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** A subquery is a `SELECT` nested inside another query, and it comes in three shapes: scalar (one value, usable in `WHERE` or `SELECT`), `IN`-list (many values, for membership checks), and `FROM`-clause (a whole temporary table, must be aliased). `EXISTS`/`NOT EXISTS` check row existence directly and are immune to a dangerous trap that `NOT IN` falls into: if the subquery's result contains even one `NULL`, `NOT IN` silently returns zero rows for everyone. A CTE (`WITH name AS (...)`) is functionally identical to a `FROM`-subquery but gives it a name, making multi-step logic readable top-to-bottom — and multiple CTEs can chain together, each one allowed to reference any CTE defined above it. `UNION ALL` stacks two result sets keeping duplicates (fast); `UNION DISTINCT` stacks them and removes exact duplicates (slower); BigQuery requires writing one or the other explicitly.

**KR:** 서브쿼리는 다른 쿼리 안에 중첩된 `SELECT`이며, 세 가지 형태가 있습니다: 스칼라(값 하나, `WHERE`나 `SELECT`에서 사용), `IN` 목록(여러 값, 포함 여부 확인용), `FROM`절(테이블 전체, 별칭 필수). `EXISTS`/`NOT EXISTS`는 행의 존재 여부를 직접 확인하며, `NOT IN`이 빠지는 위험한 함정에서 안전합니다: 서브쿼리 결과에 `NULL`이 하나라도 있으면 `NOT IN`은 모두에게 조용히 0행을 반환합니다. CTE(`WITH name AS (...)`)는 `FROM` 서브쿼리와 기능적으로 동일하지만 이름이 있어서, 여러 단계짜리 로직을 위에서 아래로 읽히게 만듭니다 — 여러 CTE를 이어 붙일 수 있으며, 각각은 자기보다 위에 정의된 CTE를 참조할 수 있습니다. `UNION ALL`은 중복을 유지한 채 두 결과를 쌓고(빠름), `UNION DISTINCT`는 쌓은 뒤 완전 중복을 제거합니다(느림). BigQuery는 둘 중 하나를 반드시 명시해야 합니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** A subquery and a CTE are the same idea — "run this query first, then use its result as if it were a table" — and the one habit worth building today is reaching for `NOT EXISTS` over `NOT IN` by default, since `NULL` can make `NOT IN` fail silently with no error to warn you.

> **KR:** 서브쿼리와 CTE는 같은 개념입니다 — "이 쿼리를 먼저 실행하고, 그 결과를 테이블처럼 쓴다" — 오늘 들여야 할 습관 하나를 꼽자면 `NOT IN` 대신 `NOT EXISTS`를 기본으로 쓰는 것인데, `NULL`이 섞이면 `NOT IN`은 경고 하나 없이 조용히 실패할 수 있기 때문입니다.

---
# ❓ Review Questions

**Q1.** Why is `WHERE amount > (SELECT AVG(amount) FROM orders)` more reliable long-term than `WHERE amount > 52200`?  
**Q1.** 왜 `WHERE amount > (SELECT AVG(amount) FROM orders)`가 `WHERE amount > 52200`보다 장기적으로 더 신뢰할 수 있는가?

A hardcoded 52200 goes stale as data changes; the subquery recomputes the average every run.  
숫자는 한 번만 맞고, 서브쿼리는 데이터가 바뀌어도 맞습니다.

**Q2.** A customer table has a `NOT IN (SELECT customer_id FROM orders)` condition, and it unexpectedly returns zero rows even though you know some customers never ordered. What's the most likely cause, and how do you fix it?  
**Q2.** 고객 테이블에 `NOT IN (SELECT customer_id FROM orders)` 조건을 걸었는데, 분명 주문 안 한 고객이 있는데도 예상과 다르게 0행이 나왔다. 가장 가능성 높은 원인은 무엇이고, 어떻게 고치는가?

If the subquery contains a NULL, NOT IN silently returns zero rows. Switch to NOT EXISTS.  
서브쿼리의 NULL. NOT IN 대신 NOT EXISTS.

**Q3.** Why does a query using a `FROM`-clause subquery need an `AS alias` after the closing parenthesis, when a CTE doesn't need one in the same spot?  
**Q3.** `FROM` 절 서브쿼리를 쓰는 쿼리는 왜 닫는 괄호 뒤에 `AS 별칭`이 필요한데, CTE는 같은 자리에 필요 없는가?

A FROM-subquery has no name, so it needs AS alias. A CTE already got its name in WITH name AS (...).  
서브쿼리는 이름 없음 → 별칭 필수. CTE는 정의할 때 이미 이름이 있음.

**Q4.** In `WITH a AS (...), b AS (...), c AS (...)`, can `a` reference `c`? Can `c` reference `a`? Why the difference?  
**Q4.** `WITH a AS (...), b AS (...), c AS (...)`에서 `a`가 `c`를 참조할 수 있는가? `c`가 `a`를 참조할 수 있는가? 왜 이런 차이가 나는가?

a cannot reference c (forward). c can reference a (backward). CTEs only look at names defined above them.  
뒷 단계만 쓸 수 있습니다. 순서를 바꿔 정의하세요.

**Q5.** You need to combine `online_orders` and `offline_orders` into one list before aggregating, and you're confident a customer's order can never appear in both tables. Should you use `UNION ALL` or `UNION DISTINCT`, and why does it matter for performance?  
**Q5.** `online_orders`와 `offline_orders`를 집계 전에 하나로 합쳐야 하고, 한 고객의 주문이 두 테이블 모두에 있을 리 없다고 확신한다. `UNION ALL`과 `UNION DISTINCT` 중 무엇을 써야 하며, 왜 성능에 영향을 주는가?

Use UNION ALL: no duplicates to remove, so skip the expensive distinct check. UNION DISTINCT would scan every row against every other row for nothing.  
확신이 있으면 ALL. 중복을 없애야만 할 때만 DISTINCT입니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*